# 07 · Enderezado de la rotación: ¿H6 se recupera de un paciente girado?

Corrección pedida por el profesor: normalizar la rotación con los metadatos (productos punto y cruz,
artículo de Ferro Sánchez et al., MethodsX 2024) y corregir el giro del paciente.
Este notebook **no entrena**: evalúa el modelo H6 ya entrenado en los **15 casos de validación** (test no se toca).

| Corrida | Qué mide |
|---|---|
| `base` | H6 sin giro (referencia) |
| `rotmm30` / `rotmm30_norm` | paciente girado 30° en milímetros, sin y con enderezado |
| `rotmm60` / `rotmm60_norm` | paciente girado 60° en milímetros, sin y con enderezado |

**Antes de correr:** activar GPU (Colab: Entorno de ejecución → Cambiar tipo → T4 · Kaggle: Settings → Accelerator → GPU) y llenar la celda de configuración.
Tiempo aproximado con GPU: 15–25 min.

In [ ]:
!nvidia-smi -L || echo "SIN GPU: activa la GPU antes de seguir"

## 1. Configuración (lo único que hay que editar)

Se necesitan dos cosas además del repo:

1. **El checkpoint de H6**: `h6_auggeo_rot35_canon_ema_w32_e60_s42.pth`.
2. **Los datos de validación**, de una de estas dos formas:
   - **Opción A (recomendada, más rápida):** `cache_s10_val.zip`, el caché ya preparado de los 15 casos de validación (lo comparte María).
   - **Opción B:** la carpeta `data/raw` original de PENGWIN (las 3 carpetas `PENGWIN_CT_train_*`); el notebook construye el caché.

En **Colab** súbanlos a una carpeta de Drive; en **Kaggle**, como un *Dataset* privado agregado al notebook (aparece en `/kaggle/input/...`).

In [ ]:
import os
EN_KAGGLE = os.path.exists("/kaggle/working")

REPO = "pengwin-analitica-2026/proyecto2-analitica-datos"
RAMA = "feat/normalizacion-orientacion"
GITHUB_TOKEN = ""   # solo si el repo es privado: token personal con permiso de lectura (no lo suban al repo)

if EN_KAGGLE:
    CKPT      = "/kaggle/input/pengwin-h6/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth"   # <- ajustar
    CACHE_ZIP = "/kaggle/input/pengwin-h6/cache_s10_val.zip"                          # opción A ("" si no)
    RAW_DIR   = ""                                                                     # opción B ("" si no)
    TRABAJO   = "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT      = "/content/drive/MyDrive/PENGWIN/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth"  # <- ajustar
    CACHE_ZIP = "/content/drive/MyDrive/PENGWIN/cache_s10_val.zip"                         # opción A ("" si no)
    RAW_DIR   = ""                                                                          # opción B, p. ej. "/content/drive/MyDrive/PENGWIN/raw"
    TRABAJO   = "/content"

for nombre, ruta in [("checkpoint", CKPT), ("caché (A)", CACHE_ZIP), ("raw (B)", RAW_DIR)]:
    if ruta:
        print(f"{nombre:12s} {'OK ' if os.path.exists(ruta) else 'NO EXISTE'}  {ruta}")
assert os.path.exists(CKPT), "No encuentro el checkpoint: revisa CKPT"
assert (CACHE_ZIP and os.path.exists(CACHE_ZIP)) or (RAW_DIR and os.path.exists(RAW_DIR)), "Falta el caché (A) o data/raw (B)"

## 2. Código del repo (rama de la corrección)

In [ ]:
import os, subprocess
os.chdir(TRABAJO)
url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, url, "repo"], check=True)
os.chdir("repo")
!git log --oneline -3
!pip -q install SimpleITK scikit-image

## 3. Datos de validación

In [ ]:
import os, shutil
os.makedirs("outputs/s10", exist_ok=True)
shutil.copy(CKPT, "outputs/s10/")
CK = "outputs/s10/" + os.path.basename(CKPT)

if CACHE_ZIP:
    # El zip trae data/cache/s10/index.csv y data/cache/s10/cache_384/*.npz
    !unzip -q -o "{CACHE_ZIP}" -d .
else:
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/raw"):
        os.symlink(RAW_DIR, "data/raw")
    VAL = "014 019 024 027 033 044 050 057 060 067 077 078 087 095 099"
    !python src/build_case_cache.py --out-dir data/cache/s10 --sizes 384 --cases {VAL}

CD = "data/cache/s10/cache_384"
print(len([f for f in os.listdir(CD) if f.endswith(".npz")]), "casos en el caché (deben ser 15)")

## 4. Pruebas automáticas (segundos)

In [ ]:
!python -m unittest tests.test_normalizacion.TestPaso1Metadatos tests.test_normalizacion.TestPaso2GiroAxial tests.test_normalizacion.TestPaso2EnCache tests.test_evaluar_normalizado tests.test_aumento_fisico 2>&1 | tail -3

## 5. Las 5 evaluaciones (validación, con TTA)

`rotmm=X` gira al paciente X grados **en milímetros reales** (el `rot=X` anterior giraba en píxeles y, en 10 de los 15 casos,
que tienen píxeles no cuadrados, deformaba la pelvis). `--normalizar` endereza cada volumen antes del modelo.

In [ ]:
CORRIDAS = [("base", ""),
            ("rotmm30", "--stress rotmm=30"), ("rotmm30_norm", "--stress rotmm=30 --normalizar"),
            ("rotmm60", "--stress rotmm=60"), ("rotmm60_norm", "--stress rotmm=60 --normalizar")]
os.makedirs("outputs/normalizacion", exist_ok=True)
for nombre, extra in CORRIDAS:
    print(f"\n========== {nombre} ==========")
    !python src/evaluate_volumes.py --checkpoint {CK} --cache-dir {CD} --split val --tta {extra} --output outputs/normalizacion/h6_{nombre}.json 2>&1 | grep -E "casos de|caso |giro estimado|Guardado"

## 6. Tabla de resultados

In [ ]:
import json
import pandas as pd
METRICAS = ["classification_f1_macro", "classification_auc_macro", "detection_map50", "detection_iou_mean",
            "segmentation_dice_macro", "instance_dice_matched"]
filas = []
for nombre, _ in CORRIDAS:
    r = json.load(open(f"outputs/normalizacion/h6_{nombre}.json"))
    fila = {"corrida": nombre, **{m: round(r["subsets"]["todo"].get(m, float("nan")), 3) for m in METRICAS}}
    giros = r.get("giros", {})
    fila["casos_enderezados"] = sum(g["corregido"] for g in giros.values()) if giros else "-"
    filas.append(fila)
tabla = pd.DataFrame(filas).set_index("corrida")
tabla.to_csv("outputs/normalizacion/resumen_h6_normalizacion.csv")
tabla

In [ ]:
# Ángulo estimado por caso en las corridas con enderezado (lo ideal: ≈ giro de estrés + giro propio del paciente)
for nombre in ("rotmm30_norm", "rotmm60_norm"):
    g = json.load(open(f"outputs/normalizacion/h6_{nombre}.json"))["giros"]
    print(nombre, {c: v["angulo"] for c, v in g.items()})

## 7. Descargar resultados (enviárselos a María)

In [ ]:
!cd outputs && zip -q -r ../resultados_normalizacion.zip normalizacion
print("Archivo listo:", os.path.abspath("resultados_normalizacion.zip"), "(en Kaggle: pestaña Output)")
if not EN_KAGGLE:
    from google.colab import files
    files.download("resultados_normalizacion.zip")